# Multi-Agent Architectures with Swarms: Swapping Topology Without Rewriting Orchestration

## Overview

Most multi-agent tutorials teach you one topology. You wire agents into a pipeline, or a supervisor tree, or a debate, and the orchestration code you write is inseparable from the shape you chose. Changing your mind later means rewriting the plumbing.

This tutorial takes a different angle. Using [Swarms](https://github.com/kyegomez/swarms), an open source multi-agent framework, we build **one set of agents** and then run the **same agents** through four different orchestration shapes: sequential, concurrent, hierarchical, and a directed acyclic graph. Switching between them is a one-string change.

The goal is not to argue that any one topology is best. It is to make topology a variable you can test rather than a commitment you make on day one.

## Motivation

Two problems show up repeatedly when multi-agent systems move past the demo stage.

**The first is premature commitment.** You cannot tell from the outside whether a research task is better served by three specialists running in parallel and a synthesiser merging them, or by a chain where each agent refines the last. The honest answer is that it depends on the task, and you find out by measuring. If swapping the shape costs a day of refactoring, you will not measure. You will keep whatever you built first.

**The second is the coupling of agent definition to agent orchestration.** When the prompt, the tools, and the control flow all live in the same object, agents stop being reusable. A well-defined `Researcher` should be droppable into a pipeline, a parallel fan-out, or a hierarchy without modification.

Swarms addresses both by separating the `Agent` primitive from the structures that run it.

## Key Components

| Component | Role |
|---|---|
| `Agent` | The single primitive. Holds a name, a system prompt, a model, tools, and a loop budget. Knows nothing about orchestration. |
| `SequentialWorkflow` | Runs agents in order. Each agent receives the previous agent's output as context. |
| `ConcurrentWorkflow` | Runs every agent in parallel against the same task and collects the results. |
| `HierarchicalSwarm` | A director decomposes the task, delegates to workers, and synthesises what comes back. |
| `GraphWorkflow` | Explicit DAG. Nodes are agents, edges are dependencies, execution follows a topological sort. |
| `SwarmRouter` | One entry point wrapping all of the above. Selects the architecture with a `swarm_type` string. |
| `AgentRearrange` | A small flow DSL where `A -> B, C` mixes sequential and concurrent execution in one expression. |

## Agent Architecture

![Swarms multi-agent architectures](../images/swarms-multi-agent-architectures.svg)

The diagram source, for anyone who wants to modify it:

```mermaid
graph TD
    R[SwarmRouter: swarm_type] --> S[SequentialWorkflow]
    R --> C[ConcurrentWorkflow]
    R --> H[HierarchicalSwarm]
    R --> G[GraphWorkflow]

    S --> S1[Researcher] --> S2[Analyst] --> S3[Writer]

    C --> C1[Agent A]
    C --> C2[Agent B]
    C --> C3[Agent C]

    H --> H0[Director]
    H0 --> H1[Worker]
    H0 --> H2[Worker]
    H1 -.report.-> H0
    H2 -.report.-> H0

    G --> G0[Ingest]
    G0 --> G1[Branch A]
    G0 --> G2[Branch B]
    G1 --> G3[Merge]
    G2 --> G3
```

## Benefits

- **Topology becomes an experiment.** Run the same task through four shapes and compare cost, latency, and output quality.
- **Agents stay reusable.** An `Agent` defined once works in every structure.
- **Provider mixing is free.** Swarms routes through LiteLLM, so a single swarm can combine models from different vendors, which matters for ensemble patterns where correlated errors are the enemy.
- **The escape hatch is explicit.** When a prebuilt shape does not fit, `GraphWorkflow` gives you arbitrary DAGs without leaving the framework.

## Required Packages

Swarms routes model calls through LiteLLM, so any provider key works. This notebook uses OpenAI by default.

In [ ]:
!pip install -qU swarms

In [ ]:
import os
from getpass import getpass

# Any LiteLLM-compatible provider works. Swap the key and the model_name together.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

MODEL = "gpt-4o-mini"  # cheap enough to run every cell below more than once

## Implementation

### Step 1: Define the agents once

This is the whole point of the exercise, so it is worth being deliberate. These three agents carry a role and nothing else. No control flow, no knowledge of what runs before or after them, no orchestration state. That is what makes them portable across every structure in this notebook.

`max_loops=1` keeps each agent to a single pass. Raising it lets an agent iterate on its own output before returning, which is useful in production but adds noise when you are comparing topologies.

In [ ]:
from swarms import Agent


def build_agents(model: str = MODEL) -> list[Agent]:
    """Build a fresh set of role agents.

    A factory rather than module-level globals: every structure below gets clean
    agents, so conversation history from one run cannot leak into the next and
    quietly flatter the second result.
    """
    researcher = Agent(
        agent_name="Researcher",
        agent_description="Gathers facts and identifies what is actually known.",
        system_prompt=(
            "You research topics rigorously. State what is established, what is "
            "contested, and what is unknown. Prefer specifics over generalities. "
            "Be concise: at most 200 words."
        ),
        model_name=model,
        max_loops=1,
    )

    analyst = Agent(
        agent_name="Analyst",
        agent_description="Finds the implications and the second-order effects.",
        system_prompt=(
            "You analyse research findings. Identify implications, trade-offs, and "
            "second-order effects others miss. Challenge weak reasoning directly. "
            "Be concise: at most 200 words."
        ),
        model_name=model,
        max_loops=1,
    )

    writer = Agent(
        agent_name="Writer",
        agent_description="Turns analysis into something a human wants to read.",
        system_prompt=(
            "You write clear, direct prose for a technical reader. No filler, no "
            "throat-clearing. Lead with the conclusion. At most 250 words."
        ),
        model_name=model,
        max_loops=1,
    )

    return [researcher, analyst, writer]


agents = build_agents()
print([a.agent_name for a in agents])

### Step 2: Sequential, where each agent builds on the last

`SequentialWorkflow` passes each agent's output forward as context. Use it when step N genuinely needs step N-1: research feeds analysis, analysis feeds the write-up.

The cost model is worth internalising. Latency is the **sum** of all agent latencies, and context grows at every hop, so a long chain gets slow and expensive in a way a parallel fan-out does not.

In [ ]:
from swarms import SequentialWorkflow

TASK = (
    "Assess whether small specialised language models will displace large "
    "general-purpose models for production agent workloads by 2027."
)

sequential = SequentialWorkflow(agents=build_agents(), max_loops=1)
sequential_result = sequential.run(TASK)

print(sequential_result)

### Step 3: Concurrent, for independent perspectives

`ConcurrentWorkflow` gives every agent the same task at the same time and collects the results. Nothing is passed between them, which is exactly the point: the outputs stay uncorrelated.

Latency is now the **maximum** of the agent latencies rather than the sum. The trade is that nobody reconciles the disagreements. You get three opinions and a merge problem.

In [ ]:
from swarms import ConcurrentWorkflow

concurrent = ConcurrentWorkflow(agents=build_agents())
concurrent_result = concurrent.run(TASK)

print(concurrent_result)

### Step 4: Hierarchical, when the task needs decomposing

`HierarchicalSwarm` introduces a director that splits the task into subtasks, delegates them, and synthesises the responses. This is the right shape when the decomposition itself requires judgement, rather than being obvious from the task statement.

The director is an extra model call on both ends, so this costs more than the flat shapes. You are paying for the planning and the synthesis.

In [ ]:
from swarms import HierarchicalSwarm

director = Agent(
    agent_name="Director",
    agent_description="Decomposes complex tasks and delegates them to workers.",
    system_prompt=(
        "You break a task into clearly separated subtasks, assign each to the "
        "best-suited worker, then merge what returns into one coherent answer. "
        "Do not do the workers' jobs yourself."
    ),
    model_name=MODEL,
    max_loops=1,
)

hierarchical = HierarchicalSwarm(
    director=director,
    agents=build_agents(),
    max_loops=1,
)
hierarchical_result = hierarchical.run(TASK)

print(hierarchical_result)

### Step 5: SwarmRouter, the actual payoff

Everything above required a different class and a different constructor signature. `SwarmRouter` collapses that into one object where the architecture is a string.

This is what makes topology testable. The loop below runs the identical task through three shapes and gives you something to compare.

In [ ]:
import time

from swarms import SwarmRouter

SHAPES = ["SequentialWorkflow", "ConcurrentWorkflow", "MixtureOfAgents"]

comparison = {}
for shape in SHAPES:
    router = SwarmRouter(agents=build_agents(), swarm_type=shape, max_loops=1)
    started = time.perf_counter()
    output = router.run(TASK)
    comparison[shape] = {
        "seconds": round(time.perf_counter() - started, 1),
        "chars": len(str(output)),
        "output": output,
    }
    print(f"{shape:<22} {comparison[shape]['seconds']:>6}s  {comparison[shape]['chars']:>6} chars")

### Step 6: AgentRearrange, for shapes that are not prebuilt

Sometimes the topology you want is neither purely sequential nor purely parallel. `AgentRearrange` takes a flow string where `->` means "then" and `,` means "at the same time".

`Researcher -> Analyst, Writer` reads as: research first, then run analysis and writing concurrently against that research. One line, and the DSL is readable enough to survive code review.

In [ ]:
from swarms import AgentRearrange

rearrange = AgentRearrange(
    agents=build_agents(),
    flow="Researcher -> Analyst, Writer",
    max_loops=1,
)
rearrange_result = rearrange.run(TASK)

print(rearrange_result)

### Step 7: GraphWorkflow, for explicit dependencies

When the dependency structure is genuinely a graph, declare it as one. `GraphWorkflow` executes a DAG in topological order and supports per-node callbacks, which is the difference between a system you can debug and one you can only stare at.

The pattern below is fan-out then fan-in: one agent's output feeds two independent branches, and a fourth agent merges them.

In [ ]:
from swarms import Edge, GraphWorkflow, Node, NodeType

researcher, analyst, writer = build_agents()
merger = Agent(
    agent_name="Merger",
    agent_description="Reconciles parallel branches into one answer.",
    system_prompt=(
        "You receive several independent analyses. Reconcile them. Name the points "
        "where they disagree instead of averaging the disagreement away."
    ),
    model_name=MODEL,
    max_loops=1,
)

wf = GraphWorkflow()
for agent in (researcher, analyst, writer, merger):
    wf.add_node(Node(id=agent.agent_name, type=NodeType.AGENT, agent=agent))

wf.add_edge(Edge(source="Researcher", target="Analyst"))
wf.add_edge(Edge(source="Researcher", target="Writer"))
wf.add_edge(Edge(source="Analyst", target="Merger"))
wf.add_edge(Edge(source="Writer", target="Merger"))

wf.set_entry_points(["Researcher"])
wf.set_end_points(["Merger"])


def on_node_complete(node_name: str, result: str) -> None:
    print(f"  [{node_name}] done, {len(str(result))} chars")


graph_result = wf.run(task=TASK, on_node_complete=on_node_complete)
print(graph_result)

## Usage Example: picking a shape with evidence

The table below is the deliverable of this whole notebook. It is not a benchmark, it is a habit: before committing to a topology, run the task through the candidates and look at what actually comes back.

Judge the outputs yourself. Latency and length are cheap to measure and easy to over-trust.

In [ ]:
print(f"{'Architecture':<24}{'Seconds':>9}{'Chars':>9}")
print("-" * 42)
for shape, row in comparison.items():
    print(f"{shape:<24}{row['seconds']:>9}{row['chars']:>9}")

print("\n" + "=" * 70)
for shape, row in comparison.items():
    print(f"\n### {shape}\n")
    print(str(row["output"])[:600], "...")

## Comparison with a single agent

The honest baseline for any multi-agent system is one good agent with a good prompt. Multi-agent architectures are not free: they cost more tokens, more latency, and more failure modes. They earn their place when the task has genuinely separable parts or benefits from independent perspectives.

Run this and compare it against the outputs above before concluding that the swarm was necessary.

In [ ]:
solo = Agent(
    agent_name="Solo",
    system_prompt=(
        "You are a rigorous analyst and a clear writer. Research the question, "
        "analyse the implications, and deliver a tight written answer. At most 250 words."
    ),
    model_name=MODEL,
    max_loops=1,
)

started = time.perf_counter()
solo_result = solo.run(TASK)
solo_seconds = round(time.perf_counter() - started, 1)

print(f"Single agent: {solo_seconds}s, {len(str(solo_result))} chars\n")
print(solo_result)

### What the comparison usually shows

Running this notebook across a range of tasks, a few patterns recur:

| | Single agent | Sequential | Concurrent | Hierarchical |
|---|---|---|---|---|
| **Latency** | Lowest | Sum of steps | Max of steps | Highest, director on both ends |
| **Token cost** | Lowest | Grows at each hop as context accumulates | Linear in agent count | Highest |
| **Best for** | Well-scoped single tasks | Genuine step dependencies | Independent perspectives | Tasks needing decomposition judgement |
| **Main failure mode** | No self-correction | An early error propagates and compounds | Nobody reconciles disagreements | Director becomes the bottleneck and the single point of failure |

The most common real finding is that **sequential chains amplify early mistakes**. If the Researcher hallucinates a fact, the Analyst treats it as given and the Writer states it with confidence. Concurrent and graph shapes are more robust here precisely because the branches do not see each other's errors.

## Additional Considerations

**Cost scales faster than you expect.** A four-agent hierarchy with `max_loops=2` is not eight model calls, it is eight plus director planning and synthesis on every loop. Set `max_loops` explicitly and avoid `"auto"` in production unless you have a hard stopping condition.

**Give every agent a distinct `agent_name`.** Swarms keys persistent memory on the agent name. Duplicate names cause agents to share and corrupt each other's memory files.

**Long sessions need context management.** For autonomous runs, set `context_length` and leave `context_compression=True` so history is summarised before it hits the context wall.

**Mix providers deliberately.** In `MixtureOfAgents`, the value comes from decorrelated errors. Three workers on the same model and prompt produce three correlated answers, and the aggregator has nothing to reconcile. Vary the model, the prompt, or both.

**Structure does not fix a bad prompt.** If a single agent fails because the instructions are vague, five agents will fail five times in parallel. Fix the prompt first, then reach for a topology.

## References

- [Swarms on GitHub](https://github.com/kyegomez/swarms)
- [Swarms documentation](https://docs.swarms.ai)
- Du et al., [Improving Factuality and Reasoning in Language Models through Multiagent Debate](https://arxiv.org/abs/2305.14325), 2023
- Wang et al., [Mixture-of-Agents Enhances Large Language Model Capabilities](https://arxiv.org/abs/2406.04692), 2024
- Wu et al., [AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation](https://arxiv.org/abs/2308.08155), 2023